# 05 · Вектор блокировки

Здесь вектор особенно наглядный. Берём только ситуации с нарушением, снимаем активации среднего слоя
на эталоне `BLOCK …` и на плохом ответе `PASS`, вычитаем средние: $v = \mu_{\text{BLOCK}} - \mu_{\text{PASS}}$.
Прибавляем $\alpha v$ к скрытым состояниям при генерации. Положительное $\alpha$ должно поднимать
полноту блокировок вместе с ложными, отрицательное — опускать обе. Это ручка порога без обучения.

In [ ]:
import sys
sys.path.insert(0, "../..")

from src import infer
from src import filter as F

from contextlib import contextmanager

import torch

model, tokenizer = infer.load_model()
dev = [r for r in F.load("dev") if r["label"] == "BLOCK"]
layers = model.model.language_model.layers
LAYER = len(layers) // 2


def mean_activation(texts):
    captured = []
    handle = layers[LAYER].register_forward_hook(
        lambda mod, args, out: captured.append((out[0] if isinstance(out, tuple) else out).float().mean(dim=1).squeeze(0).cpu()))
    try:
        with torch.no_grad():
            for text in texts:
                model(**tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device))
    finally:
        handle.remove()
    return torch.stack(captured).mean(dim=0)


def rendered(rows, key):
    return [tokenizer.apply_chat_template(r["prompt"] + r[key], tokenize=False, enable_thinking=False) for r in rows]


vector = mean_activation(rendered(dev, "chosen")) - mean_activation(rendered(dev, "rejected"))
torch.save({"vector": vector, "layer": LAYER}, F.RUNS / "steering.pt")
print(f"слой {LAYER}, ситуаций {len(dev)}, норма разности {vector.norm():.2f}")

In [ ]:
@contextmanager
def steered(alpha):
    shift = (alpha * vector).to(model.device)

    def hook(mod, args, out):
        hidden = out[0] if isinstance(out, tuple) else out
        shifted = hidden + shift.to(hidden.dtype)
        return (shifted, *out[1:]) if isinstance(out, tuple) else shifted

    handle = layers[LAYER].register_forward_hook(hook)
    try:
        yield
    finally:
        handle.remove()


for alpha in (-1.0, 1.0, 2.0):
    with steered(alpha):
        F.evaluate(model, tokenizer, f"steer{alpha:+.0f}", note=f"steering, alpha {alpha}")
F.show()